In [1]:
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
import os
import json
import pandas as pd
from augment import find_xrd_files, fetch_alpss_metrics, load_csv_numpy
import io

### Loading Environment Variables

This notebook begins by loading environment variables from a local .env file located in the same directory.
These variables include:

- GIRDER_API_URL – the base URL of the Girder API
- HTMDEC_GIRDER_API_KEY – your personal API key for the data portal
   - You can generate an API key by logging into the data portal and opening your account settings.
- IDs used later for form queries and folder lookups

Using a .env file keeps sensitive information (like API keys) out of shared notebooks while still making them easily accessible during development.

In [2]:
# --- Load environment variables from .env ---
load_dotenv()

GIRDER_API_URL = os.getenv("GIRDER_API_URL")
API_KEY = os.getenv("HTMDEC_GIRDER_API_KEY")
ALPSS_RESULTS_FOLDER = "692da3649ca0e7d06ecd3d61"
ALPSS_FORM_ID = '6931adb5bb4e23b1e86a1ff1'

### Using the REST API

The [Girder Client](https://girder.readthedocs.io/en/latest/python-client.html) provides a convenient Python interface for interacting with the data portal’s REST API. In the next cell, we: 
1. Import **GirderClient**, the official Python/CLI interface for programmatic access to Girder.
2. Create a client instance by providing the API endpoint defined earlier in the environment variables.
3. Authenticate using your API key. 
   - We store the API key in the .env file to avoid exposing it in shared code.

In [3]:
from girder_client import GirderClient
client = GirderClient(apiUrl=GIRDER_API_URL)
client.authenticate(apiKey=API_KEY)
user = client.get("user/me")
print(f"✅ Authenticated as {user['login']}")

✅ Authenticated as arachid1


# XRD

In the next cell, we search the data portal for all XRD outputs associated with a specific IGSN. Each sample in the data portal may have multiple XRD scans. <br/> 
The first cell queries all XRD .csv files attached to the IGSN, returning a dictionary where:
- keys = filenames (e.g., scan_point_0_xrd.csv)
- values = direct Girder links to the items

In [4]:
igsn_xrd_links = find_xrd_files(igsn='JHAMAD00001', client=client)
print(json.dumps(igsn_xrd_links, indent=3))

{
   "scan_point_0_xrd.csv": "https://data.htmdec.org/#item/6931bbe9a27c05f33c328dcd",
   "scan_point_1_xrd.csv": "https://data.htmdec.org/#item/6931bc9dbb4e23b1e86a261d",
   "scan_point_2_xrd.csv": "https://data.htmdec.org/#item/6931bca1bb4e23b1e86a2623",
   "scan_point_3_xrd.csv": "https://data.htmdec.org/#item/6931bca56edf207f78581cfb",
   "scan_point_4_xrd.csv": "https://data.htmdec.org/#item/6931bca86edf207f78581d01"
}


The X-ray diffraction (XRD) files you load contain Intensity vs. 2θ (Two Theta, in degrees) measurements. <br>
Once we pick one of the returned files, we load it as a numeric NumPy array directly from the portal for downstream analysis. 

In [5]:
igsn_xrd_csv = load_csv_numpy(client, igsn_xrd_links['scan_point_0_xrd.csv'])
print(igsn_xrd_csv.shape)

(1000, 2)


# ALPSS

In this step, we query the ALPSS form entries and processed output for a given PDV filename. This returns:

- Flyer Velocity — the velocity at maximum compression recorded in the ALPSS form (called ALPSS Results)
- Spall Strength — the computed spall strength value recorded in the ALPSS form
- Velocity Time History Link — a link to the processed velocity trace stored in Girder

These values allow us to analyze both the scalar metrics and the full velocity-time curve.

In [6]:
velocity_time_history, flyer_velocity, spall_strength = fetch_alpss_metrics('C1--20250605--00087', ALPSS_FORM_ID, ALPSS_RESULTS_FOLDER, client)
print("Flyer Velocity:", flyer_velocity)
print("Spall Strength:", spall_strength)

[INFO] Extracting metrics for: C1--20250605--00087
[INFO] Extracting velocity trace for: C1--20250605--00087
Flyer Velocity: 905.7013045502234
Spall Strength: 19152542751.358006


Once again, we load the array. The velocity time history file contains the particle velocity as a function of time  measured during a PDV (Photon Doppler Velocimetry) experiment.

In [7]:
igsn_velocity_history_csv = load_csv_numpy(client, velocity_time_history)
print(igsn_velocity_history_csv.shape)

(19199, 2)


# Curation Example 

#### Load the Curation Spreadsheet

This cell loads the master curation spreadsheet (test.xlsx) containing sample identifiers and associated metadata.
We read it into a pandas DataFrame so we can enrich it with links and derived metrics from the data portal.

In [8]:
path = "test.xlsx"
df = pd.read_excel(path)

#### Add XRD File Links for Each Sample using IGSNs

Here we enrich the DataFrame by writing to an existing (or new column) containing XRD scan files associated with each sample.

For every Sample_ID, we call find_xrd_files, which searches the portal for all XRD CSVs linked to that sample’s IGSN.
The result is stored as a dictionary of filenames → portal links.

This allows downstream users to click directly through to the raw diffraction data.

In [9]:
df["Sample microstructure/material characterization (EBSD/XRD images)"] = df["Sample_ID"].apply(
    lambda x: find_xrd_files(x, client)
)

#### Extract ALPSS Metrics from PDV File Names

This cell adds data to three existing/new columns to the DataFrame:

- velocity-time history — link to the PDV velocity trace CSV
- Flyer velocity — scalar extracted from the ALPSS form
- spall strength — scalar extracted from the ALPSS form

Each row’s PDV_FileName is passed to fetch_alpss_metrics, which retrieves the corresponding information

In [10]:
df[[
    "velocity-time history",
    "Flyer velocity",
    "spall strength"
]] = df['PDV_FileName'].apply(
    lambda fn: fetch_alpss_metrics(fn, ALPSS_FORM_ID, ALPSS_RESULTS_FOLDER, client)
).apply(pd.Series)

[INFO] Extracting metrics for: C1--20250605--00088
[INFO] Extracting velocity trace for: C1--20250605--00088
[INFO] Extracting metrics for: C1--20250605--00087
[INFO] Extracting velocity trace for: C1--20250605--00087
[INFO] Extracting metrics for: NONEXISTENT_PDV
[INFO] Extracting velocity trace for: NONEXISTENT_PDV
[INFO] Extracting metrics for: PDV data should be post-processed to report velocity-time history, spall strength, and HEL values by using these hyper-parameters and the characterization function from Piyush. 
[INFO] Extracting velocity trace for: PDV data should be post-processed to report velocity-time history, spall strength, and HEL values by using these hyper-parameters and the characterization function from Piyush. 


Finally, we export the enriched DataFrame to a new file, augmented_output.xlsx.
This output file contains both the original columns and all added metadata, links, and computed ALPSS metrics, making it ready for sharing or downstream analysis.

In [11]:
df.to_excel("augmented_output.xlsx", index=False)